# 04 · Between-group comparisons (Results 3.2–3.4, Figures 3–5, Table S7)

For each of the nine measures: descriptives by group; **one-way ANOVA** (F, df, p, η²) with **Tukey HSD** post hoc tests
(primary analysis); pairwise Student t-tests with Cohen's d and Hedges' g (exploratory); Welch's t, Mann–Whitney U,
Games–Howell, Welch ANOVA and Kruskal–Wallis (robustness); Holm and Benjamini–Hochberg adjustment across the
27 pairwise tests. Differences and effect sizes are *first group − second group* (e.g., Control − Suicidal).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.multitest import multipletests
import sys
sys.path.insert(0, "..")          # dbiat_analysis.py is in the folder above notebooks/
import dbiat_analysis as A

cfg = A.load_settings()
LAB = cfg["measures"]
df = pd.read_csv(A.path(cfg, "derived", "scores.csv"))
pd.set_option("display.width", 250, "display.max_columns", 40)

## 1. Tests

In [ ]:
desc, anova, pair = [], [], []
for m in A.MEASURES:
    desc.append(A.describe(df, m).reset_index().assign(measure=m))
    anova.append(dict(measure=m, **A.oneway_anova(df, m)))
    tk = A.tukey_hsd(df, m).set_index(["group1", "group2"])
    gh = A.games_howell(df, m).set_index(["group1", "group2"])
    for a, b in A.CONTRASTS:
        r = A.two_group_tests(df.loc[df.group == a, m], df.loc[df.group == b, m])
        pair.append(dict(measure=m, group1=a, group2=b, **r, p_tukey=tk.loc[(a, b), "p_tukey"],
                         p_games_howell=gh.loc[(a, b), "p_games_howell"]))
desc = pd.concat(desc)
anova = pd.DataFrame(anova)
pair = pd.DataFrame(pair)
pair["p_holm"] = multipletests(pair.p, method="holm")[1]
pair["p_fdr_bh"] = multipletests(pair.p, method="fdr_bh")[1]
A.write_table(desc, cfg, "descriptives")
A.write_table(anova, cfg, "anova")
A.write_table(pair, cfg, "between_group_pairwise")
anova[["measure", "F", "df1", "df2", "p", "eta2", "F_welch", "p_welch", "H", "p_kruskal", "levene_p", "brown_forsythe_p"]].round(4)

In [ ]:
pair[["measure", "group1", "group2", "mean1", "mean2", "t", "df", "p", "d", "g_ci_low", "g_ci_high",
      "p_tukey", "p_welch", "p_mwu", "p_games_howell", "p_holm", "p_fdr_bh"]].round(4)

## 2. Which procedures give p < .05?

In [ ]:
P = ["p", "p_welch", "p_mwu", "p_tukey", "p_games_howell", "p_holm", "p_fdr_bh"]
summary = pd.DataFrame({"procedure": ["Student t (exploratory)", "Welch t", "Mann-Whitney U", "Tukey HSD (primary)",
                                      "Games-Howell", "Holm (27 tests)", "Benjamini-Hochberg (27 tests)"],
                        "p < .05": [", ".join(f"{LAB[r.measure]} {r.group1[0]}-{r.group2[0]}"
                                              for r in pair[pair[c] < .05].itertuples()) or "none" for c in P]})
print("Omnibus tests with p < .05:", {k: anova.loc[anova[k] < .05, "measure"].tolist() for k in ["p", "p_welch", "p_kruskal"]})
A.write_table(summary, cfg, "robustness_summary")
summary

## 3. Figures 3–5
Style of the submitted figures. Bars show group means with t-based 95% confidence intervals. Each bracket gives the
uncorrected Student-t p-value (* p < .05) and, in parentheses, the Tukey-adjusted p-value. Drawn at 180 mm width.

In [ ]:
PR = pair.set_index(["measure", "group1", "group2"])

def panel(ax, col, title, ylabel, dec, ybot=None, fs=1.0, wrap=False, xtick=None):
    st = df.groupby("group")[col].agg(["mean", "std", "count"]).reindex(A.GROUPS)
    st["ci"] = stats.t.ppf(.975, st["count"] - 1) * st["std"] / np.sqrt(st["count"])
    ax.bar(A.GROUPS, st["mean"], yerr=st["ci"], capsize=5, color=["blue", "green", "red"])
    hi = float((st["mean"] + st["ci"]).max())
    lo = float(min(0, (st["mean"] - st["ci"]).min())) if ybot is None else ybot
    span = hi - lo
    for i, (m, c) in enumerate(zip(st["mean"], st["ci"])):
        y, va = (m + c + span * 0.02, "bottom") if m >= 0 else (m - c - span * 0.02, "top")
        ax.text(i, y, f"{m:.{dec}f}", ha="center", va=va, fontsize=12 * fs)
    base, step = max(hi, 0) + span * 0.14, span * (0.21 if wrap else 0.13)
    for k, (a, b) in enumerate([("Control", "Depressed"), ("Depressed", "Suicidal"), ("Control", "Suicidal")]):
        r = PR.loc[(col, a, b)]
        x1, x2 = A.GROUPS.index(a), A.GROUPS.index(b)
        y = base + k * step
        ax.plot([x1 + 0.03, x1 + 0.03, x2 - 0.03, x2 - 0.03], [y, y + span * 0.02, y + span * 0.02, y], color="black")
        ax.text((x1 + x2) / 2, y + span * 0.03,
                f"p={r.p:.3f}{'*' if r.p < .05 else ''}{chr(10) if wrap else ' '}(Tukey {r.p_tukey:.3f})",
                ha="center", va="bottom", fontsize=11 * fs)
    ax.set_ylim(lo - span * 0.12 if ybot is None else ybot, base + 3 * step + span * 0.02)
    if ybot is None:
        ax.axhline(0, color="black", lw=0.8)
    ax.set_title(title, fontsize=14 * fs)
    ax.set_ylabel(ylabel, fontsize=13 * fs)
    ax.tick_params(axis="x", labelsize=xtick or 13 * fs)
    if wrap:
        plt.setp(ax.get_xticklabels(), rotation=25, ha="right", rotation_mode="anchor")
    ax.tick_params(axis="y", labelsize=max(8.5, 10 * fs))

FIGS = {
    "Fig3": ([("life_rt", "Life:Me RT by Group", "Reaction Time (RT in msec)", 2, 600, 0.78),
              ("death_rt", "Death:Me RT by Group", "Reaction Time (RT in msec)", 2, 600, 0.78)], 3.5),
    "Fig4": ([("life_er", "Life:Me ER by Group", "Error Rate (ER)", 3, 0, 0.78),
              ("death_er", "Death:Me ER by Group", "Error Rate (ER)", 3, 0, 0.78)], 3.5),
    "Fig5": ([("D_RT", "RT-based D-score\n($D_{RT}$) by Group", "$D_{RT}$ Score", 2, None, 0.76, True, 8.6),
              ("D_ER", "ER-based D-score\n($D_{ER}$) by Group", "$D_{ER}$ Score", 3, None, 0.76, True, 8.6),
              ("D_Composite", "Composite RT and ER-based\nD-score ($D_{Composite}$) by Group",
               "$D_{Composite}$ Score", 2, None, 0.76, True, 8.6)], 4.0),
}
with plt.style.context("default"):
    for name, (panels, height) in FIGS.items():
        fig, axes = plt.subplots(1, len(panels), figsize=(180 / 25.4, height))
        for ax, spec in zip(axes, panels):
            panel(ax, *spec)
        fig.tight_layout(pad=0.4)
        A.save_fig(fig, cfg, name)
        plt.show()
        A.finalize_tiff(cfg, name)